In [ ]:
%pip install sacrebleu 
%pip install rouge-score
%pip install sentence-transformers
%pip install numpy

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
import sacrebleu

In [ ]:
df = pd.read_csv("G:\Resume-Matcher\experiments\prompts\results\responses.csv")
df.head()  

In [ ]:
def bleu_score(pred, ref):
    return sacrebleu.sentence_bleu(pred, [ref]).score

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
def rougeL_score(pred, ref):
    return rouge.score(ref, pred)['rougeL'].fmeasure

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_sim(pred, ref):
    e1 = embedder.encode(pred, convert_to_tensor=True)
    e2 = embedder.encode(ref, convert_to_tensor=True)
    return float(util.cos_sim(e1, e2))


In [ ]:
def evaluate_row(row):
    ref = row["ground_truth"]

    # Zero-shot
    bleu_z = bleu_score(row["zero_text"], ref)
    rouge_z = rougeL_score(row["zero_text"], ref)
    sem_z = semantic_sim(row["zero_text"], ref)

    # Few-shot
    bleu_f = bleu_score(row["few_text"], ref)
    rouge_f = rougeL_score(row["few_text"], ref)
    sem_f = semantic_sim(row["few_text"], ref)

    # Advanced
    bleu_a = bleu_score(row["adv_text"], ref)
    rouge_a = rougeL_score(row["adv_text"], ref)
    sem_a = semantic_sim(row["adv_text"], ref)

    return pd.Series({
        "bleu_zero": bleu_z, "rouge_zero": rouge_z, "semantic_zero": sem_z,
        "bleu_few": bleu_f, "rouge_few": rouge_f, "semantic_few": sem_f,
        "bleu_adv": bleu_a, "rouge_adv": rouge_a, "semantic_adv": sem_a,
    })

results = df.apply(evaluate_row, axis=1)
final_df = pd.concat([df, results], axis=1)
final_df.to_csv("G:\Resume-Matcher\experiments\prompts\results\evaluation_results.csv", index=False)
final_df.head()